# 使用 OpenAI 嵌入、LangChain 和 Oracle AI Database 构建语义搜索

本 cookbook 展示如何使用以下组件构建语义搜索工作流：

- **OpenAI 嵌入**：将文本转换为向量表示
- **LangChain 的 Oracle 向量存储集成**：通过熟悉的 Python 接口写入和查询向量
- **Oracle AI Database Vector Search**：将嵌入与关系型/应用数据存储在一起

OpenAI 用于生成嵌入，LangChain 提供向量存储抽象，Oracle AI Database 提供向量存储和相似度搜索。这样可以明确 OpenAI 的角色，而不会暗示 OpenAI 提供了单独的托管向量搜索功能。

这种模式适用于检索增强生成（RAG）、内部语义搜索，以及源数据已经存放在 Oracle 中的应用。它让你无需引入单独的向量数据库就能添加语义检索，同时仍可在更大的应用工作流中使用 LangChain 的检索接口。

## 架构概览

1. 使用 `text-embedding-3-small` 为文本文档生成嵌入。
2. LangChain 将文本、元数据和向量写入 Oracle AI Database 表中。
3. 使用同一个 OpenAI 模型为自然语言查询生成嵌入。
4. Oracle AI Database Vector Search 返回最接近的已存储文档。
5. 在更大的 RAG 应用中，可以将检索到的文本传给生成模型。

## 本 notebook 演示的内容

1. 从环境变量或本地 `.env` 文件加载凭据。
2. 使用 Python `oracledb` 驱动连接 Oracle AI Database。
3. 初始化 OpenAI 嵌入和 LangChain 的 Oracle 向量存储。
4. 插入一个可重复运行的小型示例数据集，且不会累积重复数据。
5. 使用 `similarity_search(..., k=3)` 运行语义相似度搜索。

## 要求

- Python 3.10+
- 具备嵌入访问权限的 OpenAI API 密钥
- 已启用 Vector Search 的 Oracle AI Database
- Python 软件包：`langchain`、`langchain-openai`、`langchain-oracledb`、`oracledb`、`python-dotenv`、`numpy`
    

### 安装依赖

如果你的环境中已经安装了这些软件包，这个单元不会做任何更改。
        

In [1]:
%%capture
%pip install -q langchain langchain-oracledb langchain-openai oracledb python-dotenv numpy
            

### 配置凭据和 Oracle 连接

本 notebook 从环境变量读取配置。你可以在 shell、notebook 环境中设置它们，也可以在本 notebook 旁边的本地 `.env` 文件中设置：

```env
OPENAI_API_KEY=your-openai-api-key
ORACLE_USER=your-oracle-user
ORACLE_PASSWORD=your-oracle-password
ORACLE_DSN=host:port/service_name
```

对于本地实验，一种常见的选择是使用 Oracle Database Free 容器，暴露 `1521` 端口，并使用 `FREEPDB1` 这样的可插拔数据库服务，此时 `ORACLE_DSN` 形如 `localhost:1521/FREEPDB1`。你也可以使用 Oracle Autonomous Database 或其他已启用 Vector Search 的 Oracle AI Database 实例。

Oracle 的 Python 驱动可以在 **Thin 模式**下运行，无需单独的 Oracle 客户端库。如果安装了 Oracle Instant Client，同一驱动还可以选择使用 **Thick 模式**，以兼容某些高级客户端功能。下面的设置单元会在可用时尝试启用 Thick 模式，否则继续使用 Thin 模式。
        

### Oracle 设置选项

本 notebook 假设你已经可以访问一个 Oracle AI Database 端点。以下任意一种设置方式都可以：

- **本地开发：** 在容器中运行或本地安装的 Oracle Database Free，使用 `FREEPDB1` 之类的服务名。
- **托管云服务：** 已启用 Vector Search 的 Oracle Autonomous Database。
- **现有环境：** 由你的团队提供的内部 Oracle AI Database 实例。

本 notebook 只需要通过 `ORACLE_DSN` 提供一个 SQL 连接字符串，不会为你创建数据库实例。
            

In [2]:
import os
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings(
    "ignore",
    message="Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.*",
    category=UserWarning,
)

import numpy as np
import oracledb
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_oracledb.vectorstores import OracleVS
from langchain_oracledb.vectorstores.oraclevs import DistanceStrategy

load_dotenv()

# Optional: uncomment and fill these in if you are not using environment variables
# or a local .env file.
# os.environ["OPENAI_API_KEY"] = "your-openai-api-key"
# os.environ["ORACLE_USER"] = "your-oracle-user"
# os.environ["ORACLE_PASSWORD"] = "your-oracle-password"
# os.environ["ORACLE_DSN"] = "host:port/service_name"

required_env_vars = [
    "OPENAI_API_KEY",
    "ORACLE_USER",
    "ORACLE_PASSWORD",
    "ORACLE_DSN",
]

missing = [name for name in required_env_vars if not os.getenv(name)]
if missing:
    raise EnvironmentError(
        "Missing required environment variables: "
        f"{missing}. Set them in your environment or in a local .env file."
    )

try:
    oracledb.init_oracle_client()
    print("Oracle Client initialized in Thick mode")
except Exception:
    print("Oracle Client running in Thin mode")

print("Configuration loaded")
            

Oracle Client initialized in Thick mode
Configuration loaded


### 连接 Oracle AI Database

LangChain 的 Oracle 向量存储会使用这个连接来创建或复用演示表、插入嵌入并运行相似度搜索。
        

In [3]:
conn = oracledb.connect(
    user=os.getenv("ORACLE_USER"),
    password=os.getenv("ORACLE_PASSWORD"),
    dsn=os.getenv("ORACLE_DSN"),
)

print("Connected to Oracle AI Database")
        

Connected to Oracle AI Database


### 选择演示表和距离度量

本演示使用一张表 `LANGCHAIN_DEMO_VECTORS` 和余弦距离。余弦距离是文本嵌入的常用默认选择，因为它比较的是向量的方向，而不是原始的模长。
            

### 初始化嵌入和 Oracle 向量存储

我们在设置向量存储的地方附近初始化 OpenAI 嵌入模型，因为 LangChain 在插入文档和搜索自然语言查询时都会用到嵌入函数。

如果已经存在一个由本 notebook 早期版本创建的不兼容演示表，就会将其删除，以便 LangChain 按其预期的列重新创建：`ID`、`TEXT`、`METADATA` 和 `EMBEDDING`。
        

In [4]:
table_name = "LANGCHAIN_DEMO_VECTORS"
distance_strategy = DistanceStrategy.COSINE

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

required_columns = {"ID", "TEXT", "METADATA", "EMBEDDING"}

with conn.cursor() as cursor:
    cursor.execute(
        """
        SELECT column_name
        FROM user_tab_columns
        WHERE table_name = :table_name
        """,
        table_name=table_name.upper(),
    )
    existing_columns = {row[0] for row in cursor.fetchall()}

if existing_columns and not required_columns.issubset(existing_columns):
    with conn.cursor() as cursor:
        cursor.execute(f"DROP TABLE {table_name} PURGE")
    conn.commit()
    print(f"Dropped incompatible existing demo table: {table_name}")

try:
    oracle_vs = OracleVS(
        client=conn,
        embedding_function=embeddings,
        table_name=table_name,
        distance_strategy=distance_strategy,
    )
    print("Oracle Vector Store initialized")
except Exception as e:
    oracle_vs = None
    print("Oracle Vector Store not initialized.")
    print("This can happen if embedding generation is unavailable or configuration is incomplete.")
    print(e)
        

Oracle Vector Store initialized


### 插入可重复运行的示例数据

示例数据有意保持很小，以便于检查检索行为。插入之前，该单元会清空演示表。这样可以保持 notebook 的幂等性：重复运行不会累积重复行，也不会改变检索结果。
        

下一个单元中的 `DELETE FROM` 是有意为之。它只会在插入前重置演示表中的示例行，使 notebook 的重复运行结果保持确定。如果不进行重置，重新运行 notebook 会再次追加相同的文档，并可能改变 top-k 检索结果。
            

In [5]:
texts = [
    "Oracle AI Database provides native vector search.",
    "LangChain integrates Oracle AI Database using the langchain-oracledb plugin.",
    "Vector search enables semantic similarity over unstructured text.",
    "Relational AI databases can now store and search embeddings efficiently.",
]

if oracle_vs:
    try:
        with conn.cursor() as cursor:
            cursor.execute(f"DELETE FROM {table_name}")
        conn.commit()
        print("Cleared existing sample documents")

        oracle_vs.add_texts(texts)
        print("Sample documents embedded and stored")
    except Exception as e:
        oracle_vs = None
        print("Skipping document embedding because vector store population failed.")
        print(e)
else:
    print("Skipping document embedding (vector store not available)")
        

Cleared existing sample documents
Sample documents embedded and stored


### 检查嵌入和表的详细信息

这些检查是可选的，并且做了保护处理，即使嵌入配额不可用，notebook 也能平稳地继续运行。表检查不会调用 OpenAI API。
        

In [6]:
sample_embedding = None
embedding_dim = None

if oracle_vs:
    try:
        sample_embedding = embeddings.embed_query("test")
        embedding_dim = len(sample_embedding)

        print("Embedding statistics:")
        print(f"- Embedding dimension: {embedding_dim}")
        print(f"- Number of documents embedded: {len(texts)}")
        print(f"- Oracle embedding table: {table_name}")

        embedding_bytes = np.array(sample_embedding).nbytes
        print("Storage size estimation:")
        print(f"- Approx. bytes per embedding: {embedding_bytes}")
        print(f"- Approx. total bytes stored: {embedding_bytes * len(texts)}")
    except Exception as e:
        print("Skipping embedding statistics because embedding generation is unavailable.")
        print(e)
else:
    print("Skipping embedding statistics (vector store not available)")

with conn.cursor() as cursor:
    cursor.execute(
        """
        SELECT column_name
        FROM user_tab_columns
        WHERE table_name = :table_name
        ORDER BY column_id
        """,
        table_name=table_name.upper(),
    )
    columns = [row[0] for row in cursor.fetchall()]

print(f"Vector table {table_name} columns: {columns}")
        

Embedding statistics:
- Embedding dimension: 1536
- Number of documents embedded: 4
- Oracle embedding table: LANGCHAIN_DEMO_VECTORS
Storage size estimation:
- Approx. bytes per embedding: 12288
- Approx. total bytes stored: 49152
Vector table LANGCHAIN_DEMO_VECTORS columns: ['ID', 'TEXT', 'METADATA', 'EMBEDDING']


### 运行语义相似度搜索

LangChain 会为自然语言查询生成嵌入，并在 Oracle 向量存储中搜索最接近的已存储文档。本示例使用 `similarity_search(..., k=3)`，它会返回最匹配的若干文档。
        

In [7]:
query = "How does Oracle support vector search?"

if oracle_vs:
    try:
        results = oracle_vs.similarity_search(query, k=3)
        print(f"LangChain similarity search returned {len(results)} results")

        for i, doc in enumerate(results, start=1):
            print(f"{i}. {doc.page_content}")
    except Exception as e:
        print("Skipping similarity search because query embedding or retrieval failed.")
        print(e)
else:
    print("Skipping similarity search (vector store not available)")
        

LangChain similarity search returned 3 results
1. Oracle AI Database provides native vector search.
2. Vector search enables semantic similarity over unstructured text.
3. Relational AI databases can now store and search embeddings efficiently.


### 检查返回的文档

LangChain 返回 `Document` 对象。在这个简单示例中，重要的字段是 `page_content`；但在更大的工作流中，可以利用元数据来跟踪来源 ID、URL、文档章节或其他应用特定的属性。
            

In [8]:
if oracle_vs and "results" in globals():
    for i, doc in enumerate(results, start=1):
        print(f"Result {i}")
        print(f"- content: {doc.page_content}")
        print(f"- metadata: {doc.metadata}")
else:
    print("Skipping document inspection because no search results are available.")
            

Result 1
- content: Oracle AI Database provides native vector search.
- metadata: {}
Result 2
- content: Vector search enables semantic similarity over unstructured text.
- metadata: {}
Result 3
- content: Relational AI databases can now store and search embeddings efficiently.
- metadata: {}


### 尝试另一个查询

同一个向量存储可以复用于更多问题，无需重新初始化嵌入模型。
        

In [9]:
if oracle_vs:
    try:
        print("Oracle AI Database vector search results:")
        results = oracle_vs.similarity_search("What is Oracle Search?", k=3)

        print(f"Number of results: {len(results)}")
        for i, doc in enumerate(results, start=1):
            print(f"{i}. {doc.page_content}")
    except Exception as e:
        print("Skipping final vector search because query embedding or retrieval failed.")
        print(e)
else:
    print("Skipping final vector search because the Oracle vector store was not initialized.")
        

Oracle AI Database vector search results:
Number of results: 3
1. Oracle AI Database provides native vector search.
2. LangChain integrates Oracle AI Database using the langchain-oracledb plugin.
3. Vector search enables semantic similarity over unstructured text.


### 结论

本 notebook 使用 OpenAI 嵌入、LangChain 和 Oracle AI Database 实现了语义向量搜索。OpenAI 将文本和查询转换为嵌入，LangChain 提供向量存储 API，Oracle AI Database 则将向量与应用数据一起存储和搜索。

同样的模式可以扩展为 RAG 工作流（将检索到的文档传给生成模型），也可以扩展为应用搜索层，让 Oracle 继续作为关系型数据和向量数据的唯一可信来源。